<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Cápitulo 2: Lidando com Palavras

Para realizar essa prática utilizaremos 2 libs: Pytorch e Tiktoken

In [71]:
import torch
import tiktoken

print(f"Pytorch versão: {torch.__version__}")
print(f"Tiktoken versão: {tiktoken.__version__}")

Pytorch versão: 2.13.0+cpu
Tiktoken versão: 0.13.0


# 2.1 Entendendo Embedding

# 2.2 Tokenizando Texto

In [72]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

In [73]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Tamanho do Texto: {len(raw_text)}\n")
print(raw_text)

Tamanho do Texto: 20479

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)

"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it's going to send the value of my picture 'way up; but I don't think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing's lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn's "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"

Vamos Picotar o Texto em Pequenos pedaços, mas antes disso ao invés de trampar com um texto deste tamanho vamos realizar isso para um texto de tamanho menor para facilitar e depois expandimos este caso.

In [74]:
import re
text = "Hello, world. O Vasco da Gama é o maior time de futebol do mundo!"
print(text)

result = re.split(r'(\s)', text)
print(result)

Hello, world. O Vasco da Gama é o maior time de futebol do mundo!
['Hello,', ' ', 'world.', ' ', 'O', ' ', 'Vasco', ' ', 'da', ' ', 'Gama', ' ', 'é', ' ', 'o', ' ', 'maior', ' ', 'time', ' ', 'de', ' ', 'futebol', ' ', 'do', ' ', 'mundo!']


Note que separamos bem as palavras, mas ainda faltou lidar com vírgulas e pontos.

In [75]:
result = re.split(r'(!|,|\.|\s)',text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'O', ' ', 'Vasco', ' ', 'da', ' ', 'Gama', ' ', 'é', ' ', 'o', ' ', 'maior', ' ', 'time', ' ', 'de', ' ', 'futebol', ' ', 'do', ' ', 'mundo', '!', '']


Ficou bom, mas ainda tem esses valores vazios no array, vamos limpar com strip.

In [76]:
# result = result.strip()
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'O', 'Vasco', 'da', 'Gama', 'é', 'o', 'maior', 'time', 'de', 'futebol', 'do', 'mundo', '!']


Agora ficou perfeito!

Voltando para aquele texto enorme que queremos repartir... Lá terão outros tipos de pontuação que não consideramos, então vamos incluir no split e realizar o picote

In [77]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:99])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera', '.', '(', 'Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence', '.', ')', '"', 'The', 'height', 'of', 'his', 'glory', '"', '--', 'that', 'was', 'what', 'the', 'women', 'called', 'it', '.', 'I', 'can', 'hear', 'Mrs', '.', 'Gideon', 'Thwing', '--', 'his', 'last', 'Chicago', 'sitter']


Agora SIM!!!

Então, agora que repartirmos estas palavras, temos quantos TOKENS ????
Vamos ver:

In [78]:
print(len(preprocessed))

4690


Temos 4690 TOKENS neste texto !

# 2.3 Converter os TOKENS em IDs

Para isso criaremos nosso vocabulário e indereçaremos um id para eles. A questão é que esse texto usado para gerar os tokens em "preprocessed" ele apresenta várias palavras repetidas, então acabariamos dando mais de um id para uma mesma palavra. Como resolver isso ?

Basta usarmos a função "set()" do python na lista de valores/palavras "preprocessed" fazendo com que duplicatas sejam deletadas e sobrem só as primeiras ocorrências (a lista virou um set e sets não permitem duplicatas). Bem prático!

In [79]:
preprocessed_sem_duplicatas = set(preprocessed)
print(type(preprocessed))
print(type(preprocessed_sem_duplicatas))
print(f"Note como reduzimos o tamanho de {len(preprocessed)} para {len(preprocessed_sem_duplicatas)} devido a redução destas duplicatas")
# print(len(preprocessed_sem_duplicatas))

<class 'list'>
<class 'set'>
Note como reduzimos o tamanho de 4690 para 1130 devido a redução destas duplicatas


Agora podemos seguir para colocar IDs em cada token único que temos:

In [80]:
preprocessed_sem_duplicatas_ordenado = sorted(preprocessed_sem_duplicatas)

In [81]:
# O livro faz isso em uma linha só do python ☠️, mas é tranquilo só seguir a ideia de usar {} e dentro colocar token:i (chave é o token e recebe o id) for i,token in enumerate(preprocessed_sem_duplicatas_ordenado)
vocab_dicionario = {}
for i,token in enumerate(preprocessed_sem_duplicatas_ordenado):
    print(f"ID: {i}")
    print(f"Token: {token}")
    vocab_dicionario[token] = i

print(vocab_dicionario)

ID: 0
Token: !
ID: 1
Token: "
ID: 2
Token: '
ID: 3
Token: (
ID: 4
Token: )
ID: 5
Token: ,
ID: 6
Token: --
ID: 7
Token: .
ID: 8
Token: :
ID: 9
Token: ;
ID: 10
Token: ?
ID: 11
Token: A
ID: 12
Token: Ah
ID: 13
Token: Among
ID: 14
Token: And
ID: 15
Token: Are
ID: 16
Token: Arrt
ID: 17
Token: As
ID: 18
Token: At
ID: 19
Token: Be
ID: 20
Token: Begin
ID: 21
Token: Burlington
ID: 22
Token: But
ID: 23
Token: By
ID: 24
Token: Carlo
ID: 25
Token: Chicago
ID: 26
Token: Claude
ID: 27
Token: Come
ID: 28
Token: Croft
ID: 29
Token: Destroyed
ID: 30
Token: Devonshire
ID: 31
Token: Don
ID: 32
Token: Dubarry
ID: 33
Token: Emperors
ID: 34
Token: Florence
ID: 35
Token: For
ID: 36
Token: Gallery
ID: 37
Token: Gideon
ID: 38
Token: Gisburn
ID: 39
Token: Gisburns
ID: 40
Token: Grafton
ID: 41
Token: Greek
ID: 42
Token: Grindle
ID: 43
Token: Grindles
ID: 44
Token: HAD
ID: 45
Token: Had
ID: 46
Token: Hang
ID: 47
Token: Has
ID: 48
Token: He
ID: 49
Token: Her
ID: 50
Token: Hermia
ID: 51
Token: His
ID: 52
Token: How

Processo de Tokenização e Mapeamento para Token IDs

O fluxo para transformar texto bruto em dados compreensíveis pelo LLM ocorre em 3 etapas principais:

1. **Tokenização (*Tokenization*):**
   - O texto bruto de entrada (*Sample text*) é fragmentado em unidades menores chamadas **tokens** (palavras, pontuações ou subpalavras).
   - *Exemplo:* `"The brown dog..."` $\rightarrow$ `['The', 'brown', 'dog', ...]`

2. **Vocabulário Existente (*Existing Vocabulary*):**
   - É o dicionário de mapeamento construído previamente com todas as palavras únicas conhecidas pelo modelo.
   - Cada palavra é associada a um número inteiro exclusivo (**ID**):
     - `'brown'` $\rightarrow$ `0`
     - `'dog'` $\rightarrow$ `1`
     - `'fox'` $\rightarrow$ `2`

3. **Mapeamento para Token IDs (*Token IDs*):**
   - Cada token do texto é substituído pelo seu respectivo número inteiro com base no vocabulário.
   - *Exemplo de saída:* `[7, 0, 1, ...]`

---
> 💡 **Resumo do Pipeline:**  
> **Texto Bruto** $\longrightarrow$ **Tokens (strings)** $\longrightarrow$ **Consulta ao Vocabulário** $\longrightarrow$ **Token IDs (inteiros)** $\longrightarrow$ *(Próxima etapa: Embeddings)*


Com esse conceito de Texto bruto -> Tokens -> Consulta vocab -> Token IDs. Podemos consolidar tudo em uma classe.

In [82]:
class Tokenizador_Simples_v1:
    def __init__(self, vocabulario):
        self.vocabulario_str_2_int = vocabulario
        self.vocabulario_int_2_str = {}
        # for i, s in enumerate(vocabulario):
        #     self.vocabulario_int_2_str[i]=s
        for string, id in vocabulario.items():
            self.vocabulario_int_2_str[id]=string
    def encoder(self, texto: str) -> list[int]: #dizendo que recebo texto string e vira lista de int
        # Objetivo é dado um texto, utilizar deste vocabulário comum para conseguir gerar Tokens IDs. Então, aqui devo pegar o Text -> Picotar -> Traduzo Token string para Token id cada item da lista de palavras

        texto_picatado_tokens_string = re.split(r'([.,;!_?"()\']|--|\s)', texto) # Com isso teremos nosso texto picotado, mas teremos espaços vazios como '' em algumas partes da lista.
        


        #para remover espaços vazios dos itemns com o primeiro "item.strip()" (vai que aguma palavra tem \n no final ou um espaço nas bordas) e só salvando o que não for espaço vazio "if item.strip()"
        texto_picatado_tokens_string = [item.strip() for item in texto_picatado_tokens_string if item.strip()]

        texto_picotado_tokens_ids =[]

        for token_string in texto_picatado_tokens_string:
            texto_picotado_tokens_ids.append(self.vocabulario_str_2_int[token_string])

        # ou com uma linha igual ele fez texto_picotado_tokens_ids = [self.vocabulario[token_string] for token_string in texto_picatado_tokens_string] ☠️

        return texto_picotado_tokens_ids

    def decoder(self, lista_texto_ids)->str: # ele quer realmente a string de volta, não apenas uma listinha
        list_texto = []
        for id in lista_texto_ids:
            list_texto.append(self.vocabulario_int_2_str[id]) # uso do vocab que criei para transformar id em str dnv
        texto_string_raw = " ".join(list_texto) # junto todas as strings em uma string só, mas cada palavra está com espaços tipó "Hello , do you like tea ?" Preciso tirar estes espaços esquisitos
        texto_string_raw = re.sub(r'\s+([,.?!"()\'])', r'\1', texto_string_raw) # crio os espaços

        return texto_string_raw
        
    

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/08.webp?123" width="500px">

In [83]:
tokenizador = Tokenizador_Simples_v1(vocab_dicionario)

# texto_qualquer = """"Vasco da gama é um dos times de todos os tempos!" Foi o que ele me disse a um tempo atrás """
texto_qualquer = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""

texto_qualquer_tokens_ids = tokenizador.encoder(texto_qualquer)
print(f"Aqui está o texto separado por tokens na forma de ids: {texto_qualquer_tokens_ids}")

Aqui está o texto separado por tokens na forma de ids: [1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [84]:
print(f"Podemos devolver o texto para sua forma padrão com : {tokenizador.decoder(texto_qualquer_tokens_ids)}")

Podemos devolver o texto para sua forma padrão com : " It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


# 2.4 Adicionando Tokens Especiais

É extremamente útil adicionar tokens que signifiquem ausência de vocabulário para uma palavra (OOV) e para simbolizar o final do texto.

- Alguns destes Tokens especiais são:
  - `[BOS]` (beginning of sequence) Marca o começo do texto
  - `[EOS]` (end of sequence) Marca aonde o texto termina. Isso é utilizado para concatenar multiplos textos não relacionados, tipo 2 artigos wikipedia diferentes...
  - `[PAD]` (padding) Se treinarmos LLMs com batchs maiores que 1. Caso incluamos alguns textos de tamanhos diferentes teremos problemas ao processa-los, para isso usamoso 'pad' no texto de menor tamanho e preenchemos ele até ficar com tamanho igual ao de maior tamanho.
- `[UNK]` representar palavras que estão fora do vocabulario

- Note que GPT-2 não precisa de nenhuma desses tokens mencionados acima, mas utiliza o `<|endoftext|>` para reduzir complexidade
- O `<|endoftext|>` é analogo ao `[EOS]` mencionado acima
- O GPT também utiliza o `<|endoftext|>` para preenchimento (*padding*) (já que normalmente usamos uma máscara ao treinar com entradas em lotes (*batches*), o modelo não presta atenção aos tokens preenchidos de qualquer forma, então não importa quais tokens sejam usados para isso).
- O GPT-2 não usa um token `<UNK>` para palavras fora do vocabulário; em vez disso, ele usa um tokenizador baseado em *Byte-Pair Encoding* (BPE), que decompõe palavras desconhecidas em unidades menores (subpalavras e caracteres), o qual discutiremos em uma seção posterior.

- Usamos o `<|endoftext|>` entre duas fontes de texto independentes:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/10.webp" width="500px">

Vamos ver o que acontece ao tentarmos tokenizar o texto abaixo:

In [85]:
tokenizer = Tokenizador_Simples_v1(vocab_dicionario)

text = "Vasco da Gama!"

tokenizer.encoder(text)

KeyError: 'Vasco'

- Note que deu erro, mas pq ? A palavra "Vasco" (Gama tbm não estará) não está dentro do nosso vocabulário dando o problema de OOV.
- Para lidarmos com estes casos nós adicionamos tokens especiais como `"<|unk|>"` para representar palavras desconhecidas no vocabulário
- Já que estamos cogitando aumentar o vocabulário, vamos adicionar logo o `"<|end of text|>"` de uma vez para o GPT-2. Para representar tanto o fim de textos quanto concatenar textos independentes

In [92]:
preprocessed_sem_duplicatas_ordenado_expandido = sorted(list(set(preprocessed))) #elimino duplicatas virando um set, dps desviro voltando para lista, ordeno e finalmente poderei dar um extend
preprocessed_sem_duplicatas_ordenado_expandido.extend(["<|endoftext|>", "<|unk|>"]) # adicionamos os tokens novos

print(preprocessed_sem_duplicatas_ordenado_expandido)

#refazendo aquele vocabulario dicionario
vocab_dicionario_expandido = {}
for id, token_str in enumerate(preprocessed_sem_duplicatas_ordenado_expandido):
    vocab_dicionario_expandido[token_str] = id
print(vocab_dicionario_expandido)

print(f"Note que aumentamos o tamanho do vocabulário ao adicionar esses tokens especiais o tamanho foi de {len(vocab_dicionario)} para {len(vocab_dicionario_expandido)}")

print(f"Podemos verificar os últimos 5 tokens {list(vocab_dicionario_expandido.items())[-5:]}")

['!', '"', "'", '(', ')', ',', '--', '.', ':', ';', '?', 'A', 'Ah', 'Among', 'And', 'Are', 'Arrt', 'As', 'At', 'Be', 'Begin', 'Burlington', 'But', 'By', 'Carlo', 'Chicago', 'Claude', 'Come', 'Croft', 'Destroyed', 'Devonshire', 'Don', 'Dubarry', 'Emperors', 'Florence', 'For', 'Gallery', 'Gideon', 'Gisburn', 'Gisburns', 'Grafton', 'Greek', 'Grindle', 'Grindles', 'HAD', 'Had', 'Hang', 'Has', 'He', 'Her', 'Hermia', 'His', 'How', 'I', 'If', 'In', 'It', 'Jack', 'Jove', 'Just', 'Lord', 'Made', 'Miss', 'Money', 'Monte', 'Moon-dancers', 'Mr', 'Mrs', 'My', 'Never', 'No', 'Now', 'Nutley', 'Of', 'Oh', 'On', 'Once', 'Only', 'Or', 'Perhaps', 'Poor', 'Professional', 'Renaissance', 'Rickham', 'Riviera', 'Rome', 'Russian', 'Sevres', 'She', 'Stroud', 'Strouds', 'Suddenly', 'That', 'The', 'Then', 'There', 'They', 'This', 'Those', 'Though', 'Thwing', 'Thwings', 'To', 'Usually', 'Venetian', 'Victor', 'Was', 'We', 'Well', 'What', 'When', 'Why', 'Yes', 'You', '_', 'a', 'abdication', 'able', 'about', 'above',

- Bem, não basta só adicionar os tokens especiais. Devemos adicionar restrições na nossa classe para os casos deles.

In [94]:
class Tokenizador_Simples_v2:
    def __init__(self, vocabulario):
        self.vocabulario_str_2_int = vocabulario
        self.vocabulario_int_2_str = {}
        # for i, s in enumerate(vocabulario):
        #     self.vocabulario_int_2_str[i]=s
        for string, id in vocabulario.items():
            self.vocabulario_int_2_str[id]=string
    def encoder(self, texto: str) -> list[int]: #dizendo que recebo texto string e vira lista de int
        # Objetivo é dado um texto, utilizar deste vocabulário comum para conseguir gerar Tokens IDs. Então, aqui devo pegar o Text -> Picotar -> Traduzo Token string para Token id cada item da lista de palavras

        texto_picatado_tokens_string = re.split(r'([.,;!_?"()\']|--|\s)', texto) # Com isso teremos nosso texto picotado, mas teremos espaços vazios como '' em algumas partes da lista.
        


        #para remover espaços vazios dos itemns com o primeiro "item.strip()" (vai que aguma palavra tem \n no final ou um espaço nas bordas) e só salvando o que não for espaço vazio "if item.strip()"
        texto_picatado_tokens_string = [item.strip() for item in texto_picatado_tokens_string if item.strip()]

        texto_picotado_tokens_ids =[]

        for token_string in texto_picatado_tokens_string:
            if token_string in self.vocabulario_str_2_int:
                texto_picotado_tokens_ids.append(self.vocabulario_str_2_int[token_string])
            else:
                texto_picotado_tokens_ids.append(self.vocabulario_str_2_int["<|unk|>"])

        # ou com uma linha igual ele fez texto_picotado_tokens_ids = [self.vocabulario[token_string] for token_string in texto_picatado_tokens_string] ☠️

        return texto_picotado_tokens_ids

    def decoder(self, lista_texto_ids)->str: # ele quer realmente a string de volta, não apenas uma listinha
        list_texto = []
        for id in lista_texto_ids:
            list_texto.append(self.vocabulario_int_2_str[id]) # uso do vocab que criei para transformar id em str dnv
        texto_string_raw = " ".join(list_texto) # junto todas as strings em uma string só, mas cada palavra está com espaços tipó "Hello , do you like tea ?" Preciso tirar estes espaços esquisitos
        texto_string_raw = re.sub(r'\s+([,.?!"()\'])', r'\1', texto_string_raw) # crio os espaços

        return texto_string_raw
        
    

Vamos testar agora usando OOV e 2 textos sendo juntados

In [104]:
tokenizador_v2 = Tokenizador_Simples_v2(vocab_dicionario_expandido)

texto_1 = "Vasco da gama is a team in Libertadores"
texto_2 = "In the sunlit terraces of the palace."

texto_juntado = " <|endoftext|> ".join([texto_1, texto_2])
print(texto_juntado)

texto_juntado_tokens_ids = tokenizador_v2.encoder(texto_juntado)
print(texto_juntado_tokens_ids)

print(tokenizador_v2.decoder(texto_juntado_tokens_ids))

Vasco da gama is a team in Libertadores <|endoftext|> In the sunlit terraces of the palace.
[1131, 1131, 1131, 584, 115, 1131, 568, 1131, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
<|unk|> <|unk|> <|unk|> is a <|unk|> in <|unk|> <|endoftext|> In the sunlit terraces of the <|unk|>.


# 2.5 BytePair encoding (BPE) 

- O GPT-2 utilizou o *Byte-Pair Encoding* (BPE) como o seu tokenizador.
- Ele permite que o modelo decomponha palavras que não estão em seu vocabulário predefinido em unidades menores de subpalavras (*subwords*) ou até mesmo em caracteres individuais, tornando-o capaz de lidar com **qualquer palavra fora do vocabulário** (*out-of-vocabulary*).
- Por exemplo, se o vocabulário do GPT-2 não contiver a palavra *"unfamiliarword"*, ele pode tokenizá-la como `["unfam", "iliar", "word"]` ou alguma outra divisão em subpalavras, dependendo das regras de fusão (*merges*) aprendidas no treinamento do BPE.
- O tokenizador BPE original pode ser encontrado aqui: [https://github.com/openai/gpt-2/blob/master/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- Neste capítulo, estamos utilizando o tokenizador BPE da biblioteca *open-source* [tiktoken](https://github.com/openai/tiktoken) da OpenAI, que implementa seus algoritmos centrais em Rust para melhorar a eficiência computacional.
- Criei um notebook em [./bytepair_encoder](../02_bonus_bytepair-encoder) que compara essas duas implementações lado a lado (o *tiktoken* foi cerca de 5x mais rápido no texto de exemplo).

In [105]:
import importlib #saber versão
import tiktoken #aonde vai ter o BPE em rust (+eficiente)

print("tiktoken versão:", importlib.metadata.version("tiktoken"))

tiktoken versão: 0.13.0


In [111]:
tokenizador_tiktoken = tiktoken.get_encoding("gpt2") #gera um objeto tokenizador ao carregar o vocabulário do gpt-2 50257 tokens (NUNCA terá o "unk" pq ela se vira em repartir tokens em tokens cada vez menores, mesmo que resulte me letras)
# tipo quando criei o tokenizador = Tokenizador_Simples_v2(vocab_dicionario_expandido)

text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
) # texto que o "hello" deveria ser unknow igual o "vasco"

texto_tokens_ids_tiktoken = tokenizador_tiktoken.encode(text, allowed_special={"<|endoftext|>"}) # igual eu usar o tokenizador.encoder(texto) que criamos lá em cima

print(texto_tokens_ids_tiktoken)

print("Esse 50256 é literalmente o token do endoftext")

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]
Esse 50256 é literalmente o token do endoftext


In [113]:
texto_tokens_strings_tiktoken = tokenizador_tiktoken.decode(texto_tokens_ids_tiktoken)
print(texto_tokens_strings_tiktoken)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


- O gpt-2 com certeza tinha o "Hello" diferentemente do exemplo que estavamos fazendo antes, mas a ideia é a da imagem abaixo. Dada uma palavra desconhecida ele vai picotando gradativamente a palavra até chegar numa que ele conheça, mesmo que seja uma letra. Quebra em subwords ou caracteres.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/11.webp" width="300px">

# 2.6 Amostragem de dados com janela deslizante


Basicamente queremos preparar os dados para treinar nossa LLM. A ideia da LLM é prever sempre uma palavra a frente (não inventa de pensar em mais de uma ou t+3...), então basicamente utilizaremos de uma ideia bem parecida com lá em séries temporais. Vamos separar os dados de modo a LLM ir prevendo uma palavra por vez (com base nas t palavras anteriores) e assim seguiremos treinando. Tipo a imagem abaixo:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/12.webp" width="400px">

Pegando o texto do arquivo __the-verdict.txt__ e tokenizando em ids:

In [115]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizador_tiktoken.encode(raw_text)
print(len(enc_text))

5145


- Nós queremos que para cada chunck do texto tenhamos entradas e targets
- Como o modelo deve prever o futuro, o target será a entrada deslocada 1 indice para direita

In [116]:
enc_sample = enc_text[50:]

In [118]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

print("Note que está usando os tokens_ids [290, 4920, 2241, 287] para prever o token id [257]")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]
Note que está usando os tokens_ids [290, 4920, 2241, 287] para prever o token id [257]


Mas pq passamos 4 valores então ? Pq passamos o tamanho da janela de contexto ? A ideia é que ele realize mais de 1 previsão, ele utilizará tudo até antes do token t para prever este token t

Many-to-Many igual em RNNs, estamos utilizando vários para prever vários. Igual o vídeo do coreano. Só q os transformers fazem em paralelo

Como o Modelo Aprende: Previsão do Próximo Token

Diferente de modelos clássicos de séries temporais que usam uma janela inteira para prever apenas um único valor no final, os modelos autoregressivos (como o **GPT**) treinam **todas as previsões intermediárias em paralelo** dentro da mesma janela de contexto:

Para uma janela de contexto de tamanho 4 (`context_size = 4`), temos 4 tarefas de aprendizado simultâneas:

1. **Passo 1:** `[290]` $\longrightarrow$ prevê **`4920`**  
   *(Dado `"and"`, prevê `"established"`)*

2. **Passo 2:** `[290, 4920]` $\longrightarrow$ prevê **`2241`**  
   *(Dado `"and established"`, prevê `"himself"`)*

3. **Passo 3:** `[290, 4920, 2241]` $\longrightarrow$ prevê **`287`**  
   *(Dado `"and established himself"`, prevê `"in"`)*

4. **Passo 4:** `[290, 4920, 2241, 287]` $\longrightarrow$ prevê **`257`**  
   *(Dado `"and established himself in"`, prevê `"a"`)*

---
> 💡 **Por que isso é importante?**  
> Graças à **Máscara de Atenção Causal (*Causal Attention Mask*)**, a rede neural consegue calcular o erro e atualizar os pesos para **todas essas 4 previsões de uma só vez** em uma única passada na GPU, aproveitando 100% dos dados da janela.


In [119]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


decodificando para vermos as palavras

In [120]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizador_tiktoken.decode(context), "---->", tokenizador_tiktoken.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


Isso não da data leakage pq o mecanismo de attention possui o mecanismo de Causal Mask (mascara causal) que anula os valores futuros (professor mostra no slide)

- Cuidaremos da previsão da próxima palavra em um capítulo posterior, depois de cobrirmos o mecanismo de atenção.
- Por enquanto, implementamos um carregador de dados (*data loader*) simples que itera sobre o conjunto de dados de entrada e retorna as entradas e os alvos deslocados em uma posição (*shifted by one*).

In [121]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.13.0+cpu


- Usaremos a abordagem do sliding window, deslocando a janela em +1 casa:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/13.webp?123" width="500px">

In [129]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, texto, tokenizador_gpt_dataset, tamanho_janela, stride):
        self.input_tokens_id = []
        self.target_tokens_id = []

        dados_tokenizados_id = tokenizador_gpt_dataset.encode(texto)

        for i in range(0, len(dados_tokenizados_id)-tamanho_janela, stride):
            input_chunk = dados_tokenizados_id[i: i + tamanho_janela] # os dados de entrada vão de i até i+janela
            target_chunk = dados_tokenizados_id[i+1: i + tamanho_janela+1] # os dados de entrada é só deslocar 1 no começo e fim: i+1 até i+janela+1
            self.input_tokens_id.append(torch.tensor(input_chunk)) #input_tokens deverão ser introduzidos como tensores
            self.target_tokens_id.append(torch.tensor(target_chunk))#target_tokens deverão ser introduzidos como tensores 

            
    def __len__(self):
        return len(self.input_tokens_id)

    def __getitem__(self, idx):
        return self.input_tokens_id[idx], self.target_tokens_id[idx]



In [130]:
def criar_data_loader(txt, batch_size=4 , tamanho_janela=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizador_data_loader = tiktoken.get_encoding("gpt2") # gero o tokenizador
    dataset=GPTDatasetV1(txt,tokenizador_data_loader,tamanho_janela, stride) # chamo o trem de gerar dados
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size, #tamanho do lote. Numero de amostras agrupados por tensor.
        shuffle=shuffle, # embaralhar os dados
        drop_last=drop_last, # se o numero de dados não for perfeitamente divisivel pelo tamanho de lotes ele descarta o restinho (descarta o lote incompleto)
        num_workers=num_workers # processamentos em paralelo processador
    )

    return dataloader


Processará em uma etapa(!=epoca) 1 batch:
Serão batches de tamanho [4,256]:

Linha 1: [token_1, token_2, token_3, ... até o token_256] (Frase 1)

Linha 2: [token_1, token_2, token_3, ... até o token_256] (Frase 2)

Linha 3: [token_1, token_2, token_3, ... até o token_256] (Frase 3)

Linha 4: [token_1, token_2, token_3, ... até o token_256] (Frase 4)


- vamos testar o dataloader

In [131]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [134]:
dataloader = criar_data_loader(
    raw_text, batch_size=1, tamanho_janela=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [135]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


Um exemplo usando stride igual a janela de contexto

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/14.webp" width="500px">

- Também podemos criar saídas agrupadas em lotes (*batches*).
- Observe que aumentamos o *stride* aqui para não termos sobreposição entre os lotes, pois uma sobreposição excessiva pode levar ao aumento do sobreajuste (*overfitting*).

In [136]:
dataloader = criar_data_loader(raw_text, batch_size=8, tamanho_janela=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
